In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

.................Experiment A — classification.................

In [2]:
clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [3]:
X = clf_df.drop(columns=['ticket_id', 'escalated'])
y = clf_df['escalated']

In [4]:
numeric_features = X.select_dtypes(include=['number']).columns

categorical_features = X.select_dtypes(include=['object', 'category', 'str']).columns

In [5]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [7]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [8]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [9]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

In [10]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

In [11]:
cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

In [12]:
cv_results = {}

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1'
    )

    cv_results[name] = {
        'Mean F1': scores['test_score'].mean(),
        'Std F1': scores['test_score'].std()
    }

results_df = pd.DataFrame(cv_results).T
display(results_df) 

,Mean F1,Std F1
Logistic Regression,0.392651,0.062747
KNN,0.360835,0.050295
Decision Tree,0.330666,0.058126


In [13]:
logistic_pipeline = pipelines['Logistic Regression']

In [14]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

In [15]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)

In [16]:
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1:", grid_search.best_score_)

Best Parameters: {'classifier__C': 10}
Best Cross-Validation F1: 0.4225932078257659


In [17]:
tuning_results = pd.DataFrame(grid_search.cv_results_)

display(
    tuning_results[
        [
            'param_classifier__C',
            'mean_test_score',
            'std_test_score',
            'rank_test_score'
        ]
    ].sort_values('rank_test_score')
)

,param_classifier__C,mean_test_score,std_test_score,rank_test_score
3,10.00,0.422593,0.088225,1
4,100.00,0.422593,0.088225,1
2,1.00,0.392651,0.062747,3
1,0.10,0.327873,0.046086,4
0,0.01,0.000000,0.000000,5


In [18]:
best_model = grid_search.best_estimator_

In [19]:
y_pred = best_model.predict(X_test)


In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180



In [21]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[137   5]
 [ 30   8]]


In [22]:
tn, fp, fn, tp = cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

True Negatives: 137
False Positives: 5
False Negatives: 30
True Positives: 8


.................Experiment B  regression .................

In [23]:
X_reg = reg_df.drop(columns=['ticket_id', 'resolution_time_hours'])

y_reg = reg_df['resolution_time_hours']

print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)

X_reg shape: (800, 5)
y_reg shape: (800,)


In [24]:
numeric_features_reg = X_reg.select_dtypes(include=['number']).columns

categorical_features_reg = X_reg.select_dtypes(
    include=['object', 'category', 'str']
).columns

print("Numerical:", list(numeric_features_reg))
print("Categorical:", list(categorical_features_reg))

Numerical: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical: ['priority', 'channel']


In [25]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [26]:
numeric_transformer_reg = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [27]:
categorical_transformer_reg = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [28]:
preprocessor_reg = ColumnTransformer(transformers=[
    ('num', numeric_transformer_reg, numeric_features_reg),
    ('cat', categorical_transformer_reg, categorical_features_reg)
])

In [29]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}

In [30]:
reg_pipelines = {}

for name, model in reg_models.items():
    reg_pipelines[name] = Pipeline(steps=[
        ('preprocessor', preprocessor_reg),
        ('regressor', model)
    ])

In [31]:
from sklearn.model_selection import KFold

reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [32]:
reg_cv_results = {}

for name, pipeline in reg_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring='neg_mean_absolute_error'
    )

    mae_scores = -scores['test_score']

    reg_cv_results[name] = {
        'Mean MAE': mae_scores.mean(),
        'Std MAE': mae_scores.std()
    }

reg_results_df = pd.DataFrame(reg_cv_results).T
display(reg_results_df)

,Mean MAE,Std MAE
Linear Regression,2.035134,0.171650
Ridge Regression,2.035127,0.171404
Decision Tree,3.239097,0.053807


In [33]:
ridge_pipeline = reg_pipelines['Ridge Regression']

In [34]:
ridge_param_grid = {
    'regressor__alpha': [0.01, 0.1, 1, 10, 100, 1000]
}

In [35]:
ridge_grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring='neg_mean_absolute_error',
    cv=reg_cv,
    n_jobs=-1
)

In [36]:
ridge_grid_search.fit(X_train_reg, y_train_reg)

print("Best Parameters:", ridge_grid_search.best_params_)
print("Best CV MAE:", -ridge_grid_search.best_score_)

Best Parameters: {'regressor__alpha': 0.1}
Best CV MAE: 2.035118085555478


In [37]:
ridge_tuning_results = pd.DataFrame(ridge_grid_search.cv_results_)

display(
    ridge_tuning_results[
        [
            'param_regressor__alpha',
            'mean_test_score',
            'std_test_score',
            'rank_test_score'
        ]
    ].sort_values('rank_test_score')
)

,param_regressor__alpha,mean_test_score,std_test_score,rank_test_score
1,0.10,-2.035118,0.171627,1
2,1.00,-2.035127,0.171404,2
0,0.01,-2.035133,0.171648,3
3,10.00,-2.043030,0.166698,4
4,100.00,-2.209300,0.125366,5
5,1000.00,-2.766180,0.076377,6


In [38]:
best_ridge_model = ridge_grid_search.best_estimator_

In [39]:
y_reg_pred = best_ridge_model.predict(X_test_reg)

In [40]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test_reg, y_reg_pred)

rmse = np.sqrt(mean_squared_error(y_test_reg, y_reg_pred))

print("Test MAE:", mae)
print("Test RMSE:", rmse)

Test MAE: 2.230086144180656
Test RMSE: 3.4027561084563143


In [41]:
display(y_reg.describe())

count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64

In [42]:
display(y_reg.sort_values(ascending=False).head(10))

398    27.488229
193    26.615113
188    26.135421
49     24.108510
502    24.100668
573    24.017656
51     23.929861
298    23.183383
113    18.267015
19     17.981195
Name: resolution_time_hours, dtype: float64


REQUIRED REPORT  


1. PROBLEM FRAMING

The purpose of this laboratory was to build and evaluate machine learning models for service ticket prediction. We performed two experiments.

Classification: Predict whether a service ticket will be escalated (0 = No, 1 = Yes).

Regression: Predict the number of hours required to resolve a service ticket.

The goal was to compare different algorithms, select the best-performing models, and evaluate their performance on unseen data.


2. DATA PREPROCESSING

First, I loaded and inspected both datasets. I removed ticket_id because it was only an identifier and not useful for prediction.

I separated the features into numerical and categorical groups.

For numerical features, I used median imputation to handle missing values and StandardScaler to scale the data.

For categorical features, I used most-frequent imputation and OneHotEncoder to convert categories into numerical values.

I combined these operations using ColumnTransformer and Pipeline to ensure consistent preprocessing and prevent data leakage.


3. VALIDATION STRATEGY

I divided both datasets into 80% training data and 20% testing data using random_state=42.

For classification, I used StratifiedKFold with five folds because the target classes were imbalanced. Stratification maintained approximately the same class distribution across the folds.

For regression, I used KFold with five folds because the target variable was continuous.

I used cross-validation to compare models and GridSearchCV to tune their hyperparameters.


4. CLASSIFICATION EXPERIMENT

I compared three models: Logistic Regression, KNN, and Decision Tree.

I used F1-score because the dataset was imbalanced and we wanted to balance precision and recall.

Cross-Validation Results:

Logistic Regression: Mean F1 = 0.3927, Std F1 = 0.0627

KNN: Mean F1 = 0.3608, Std F1 = 0.0503

Decision Tree: Mean F1 = 0.3307, Std F1 = 0.0581

Logistic Regression achieved the highest mean F1-score, so we selected it for tuning.

I tested C values of 0.01, 0.1, 1, 10, and 100. The best value was C = 10, with a cross-validation F1-score of 0.4226.

Final Test Results:

Accuracy = 0.81

Class 1 Precision = 0.62

Class 1 Recall = 0.21

Class 1 F1-score = 0.31

Logistic Regression was the best candidate, but its low recall showed that it missed many escalated tickets.


5. REGRESSION EXPERIMENT

I compared Linear Regression, Ridge Regression, and Decision Tree Regressor.

I used Mean Absolute Error (MAE) because it measures the average prediction error in hours.

Cross-Validation Results:

Linear Regression: Mean MAE = 2.0351, Std MAE = 0.1717

Ridge Regression: Mean MAE = 2.0351, Std MAE = 0.1714

Decision Tree: Mean MAE = 3.2391, Std MAE = 0.0538

Ridge Regression achieved the lowest mean MAE, so we selected it for tuning.

I tested alpha values of 0.01, 0.1, 1, 10, 100, and 1000. The best value was alpha = 0.1, with a cross-validation MAE of approximately 2.0351.

Final Test Results:

MAE = 2.2301 hours

RMSE = 3.4028 hours

This means the model's predictions differed from actual resolution times by approximately 2.23 hours on average.


6. LEAKAGE PREVENTION

I prevented data leakage by placing preprocessing operations inside pipelines. During cross-validation, preprocessing was fitted only on each training fold and then applied to its validation fold.

I also kept the test datasets separate from training and hyperparameter tuning. This ensured that test results provided an independent evaluation of the final models.


7. FINAL MODEL JUSTIFICATION AND LIMITATIONS

For classification, Iwe selected Logistic Regression because it achieved the highest mean cross-validation F1-score of 0.4226 after tuning.

For regression, I selected Ridge Regression because it achieved the lowest cross-validation MAE of approximately 2.0351.

However, Logistic Regression had low recall for escalated tickets, while Ridge Regression still had prediction errors. Therefore, both models could benefit from further improvement before real-world deployment.


8. REFLECTIONS

1. Why stratify the classification split?

I stratified the split to maintain approximately the same class proportions in training and testing data because the classification dataset was imbalanced.

2. Why must preprocessing be inside the pipeline?

Preprocessing must be inside the pipeline to prevent data leakage. It ensures that imputation, scaling, and encoding are fitted only on training data during each cross-validation fold.

3. What does fold variability tell you?

Fold variability shows how consistently a model performs across different folds. A lower standard deviation generally means more consistent performance.

4. Why should the test set be evaluated only once?

The test set should be evaluated only after model selection and tuning. Repeatedly using it to make decisions can cause overfitting and produce unreliable results.

5. What deployment constraints might change your choice?

Deployment decisions may depend on prediction speed, computing resources, accuracy requirements, interpretability, and the cost of incorrect predictions. For example, a company may prefer a classification model with higher recall to avoid missing escalated tickets.


9. CONCLUSION

In this laboratory, I developed and evaluated classification and regression models for service ticket prediction. Logistic Regression was selected for classification, while Ridge Regression was selected for regression.

I used cross-validation, hyperparameter tuning, pipelines, and separate test datasets to make our evaluation more reliable.

The experiments demonstrated the importance of selecting suitable metrics, preventing data leakage, and considering model limitations before deployment.
